In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# Gate G2 (spec 11, decision 23): make smoke runs the six record-size arms at 1 MiB x 1 KiB, S3 at 1 MiB and
# S4 at 64 KiB, 3 repetitions each, in under 10 minutes, on AC power with the lid open.
import json, subprocess, sys, time
from collections import Counter
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, validate

assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
before = {p.name for p in (repo / "results/raw").glob("*_smoke_*")}
t0 = time.monotonic()
r = subprocess.run(["make", "smoke"], cwd=repo, capture_output=True, text=True)
wall_min = (time.monotonic() - t0) / 60
run_dir = next(repo / "results/raw" / n for n in {p.name for p in (repo / "results/raw").glob("*_smoke_*")} - before)
print(f"make smoke: exit {r.returncode}, {wall_min:.1f} min wall, run {run_dir.name}")
print(r.stderr[-3000:])
meta = json.loads((run_dir / "meta.json").read_text())
print({k: meta[k] for k in ("git_sha_harness", "platform", "tier")})
assert meta["git_sha_harness"] and not meta["git_sha_harness"].endswith("-dirty"), "commit src/ and configs/ first"
assert r.returncode == 0
assert wall_min < 10, f"smoke took {wall_min:.1f} min, spec 11 allows 10"

make smoke: exit 0, 2.8 min wall, run 20261112T084427Z_smoke_645753
20261112T084427Z_smoke_645753: 0 of 16 done, 16 to run
[1/16] S0-go_1MiB_1KiB_offline (timeout 900 s)
    ok
[2/16] S0-go_1MiB_1KiB_online (timeout 900 s)
    ok
[3/16] S0-cpp_1MiB_1KiB_offline (timeout 900 s)
    ok
[4/16] S0-cpp_1MiB_1KiB_online (timeout 900 s)
    ok
[5/16] S1_1MiB_1KiB_offline (timeout 900 s)
    ok
[6/16] S1_1MiB_1KiB_online (timeout 900 s)
    ok
[7/16] S2_1MiB_1KiB_offline (timeout 900 s)
    ok
[8/16] S2_1MiB_1KiB_online (timeout 900 s)
    ok
[9/16] M0_1MiB_1KiB_offline (timeout 900 s)
    ok
[10/16] M0_1MiB_1KiB_online (timeout 900 s)
    ok
[11/16] M1_1MiB_1KiB_offline (timeout 900 s)
    ok
[12/16] M1_1MiB_1KiB_online (timeout 900 s)
    ok
[13/16] S3_1MiB_1bit_offline (timeout 900 s)
    ok
[14/16] S3_1MiB_1bit_online (timeout 900 s)
    ok
[15/16] S4_64KiB_1bit_offline (timeout 2700 s)
    ok
[16/16] S4_64KiB_1bit_online (timeout 2700 s)
    ok
20261112T084427Z_smoke_645753: {'ok': 16}

{

In [2]:
# Every cell's final rows: status, row count, schema problems, and correctness of the online rows.
ledger = [json.loads(l) for l in (run_dir / "ledger.jsonl").read_text().splitlines()]
print("ledger statuses:", dict(Counter(e["status"] for e in ledger)))
bad = []
for f in sorted(run_dir.glob("*_*_*_*.jsonl")):
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    probs = [str(p) for r in rows for p in validate.check_row(r, reference_platform=meta["platform"]) if p.severity == "error"]
    online = [r for r in rows if r["phase"] == "online" and r["status"] == "ok"]
    statuses = sorted({r["status"] for r in rows})
    wrong = sum(not r["correct"] for r in online)
    print(f"{f.stem:28s} {len(rows):3d} rows  {statuses}  wrong {wrong}  schema errors {len(probs)}")
    if probs or wrong or statuses != ["ok"]:
        bad.append(f.stem)
print(f"{len(bad)} cells not clean: {bad}")
assert not bad

ledger statuses: {'ok': 16}
M0_1MiB_1KiB_offline           4 rows  ['ok']  wrong 0  schema errors 0
M0_1MiB_1KiB_online            4 rows  ['ok']  wrong 0  schema errors 0
M1_1MiB_1KiB_offline           4 rows  ['ok']  wrong 0  schema errors 0
M1_1MiB_1KiB_online            4 rows  ['ok']  wrong 0  schema errors 0
S0-cpp_1MiB_1KiB_offline       4 rows  ['ok']  wrong 0  schema errors 0
S0-cpp_1MiB_1KiB_online        4 rows  ['ok']  wrong 0  schema errors 0
S0-go_1MiB_1KiB_offline        4 rows  ['ok']  wrong 0  schema errors 0
S0-go_1MiB_1KiB_online         4 rows  ['ok']  wrong 0  schema errors 0
S1_1MiB_1KiB_offline           4 rows  ['ok']  wrong 0  schema errors 0
S1_1MiB_1KiB_online            4 rows  ['ok']  wrong 0  schema errors 0
S2_1MiB_1KiB_offline           4 rows  ['ok']  wrong 0  schema errors 0
S2_1MiB_1KiB_online            4 rows  ['ok']  wrong 0  schema errors 0
S3_1MiB_1bit_offline           4 rows  ['ok']  wrong 0  schema errors 0
S3_1MiB_1bit_online            4 row

In [3]:
# The figures and tables from the smoke rows alone (G2). F8 to F11 need the session, network and load runs,
# so they are skipped here with a note.
r = subprocess.run(["make", "figures", f"RUNS=results/raw/{run_dir.name}"], cwd=repo, capture_output=True, text=True)
print(f"make figures: exit {r.returncode}")
print("\n".join(l for l in r.stderr.splitlines() if "skipped" in l))
made = sorted(p.name for p in (repo / "results/figures").iterdir() if p.is_file())
print(made)
need = [f"F{i}_" for i in range(1, 7)] + [f"T{i}_" for i in range(1, 5)]
missing = [n for n in need if not any(m.startswith(n) for m in made)]
print("missing:", missing)
assert r.returncode == 0 and not missing

make figures: exit 0
skipped F8_session: no session comparison; pass --session results/raw/<run_id>/session after session_check.py
skipped F9_emulated_network: no emulated-link rows; pass --netem results/raw/<run_id>/netem after netem.py
skipped F10_load_latency: no load-generator rows; pass --loadgen results/raw/<run_id>/loadgen after harness/loadgen.py
skipped F11_cores: no load-generator rows; pass --loadgen results/raw/<run_id>/loadgen after harness/loadgen.py
['F1_reproduction.pdf', 'F1_reproduction.png', 'F2_overhead_heatmaps.pdf', 'F2_overhead_heatmaps.png', 'F3_absolute_cost.pdf', 'F3_absolute_cost.png', 'F4_amortised.pdf', 'F4_amortised.png', 'F5_network.pdf', 'F5_network.png', 'F6_breakdown.pdf', 'F6_breakdown.png', 'T1_machine.csv', 'T1_machine.md', 'T2_scheme_params.csv', 'T2_scheme_params.md', 'T3_reproduction.csv', 'T3_reproduction.md', 'T4_overheads.csv', 'T4_overheads.md']
missing: []
